In [5]:
import requests
import time
import json
import pandas as pd

url = "https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://n.news.naver.com/article/001/0016305054" # 실제 뉴스 기사 주소
}

all_comments = []
previous_comments = []
page = 1

# 최초 요청 시 사용할 기본 파라미터 세팅
params = {
    "ticket": "news",
    "templateId": "default_politics_m3",
    "pool": "cbox5",
    "lang": "ko",
    "country": "KR",
    "objectId": "news001,0016305054", 
    "pageSize": 20,
    "indexSize": 10,
    "listType": "OBJECT",
    "page": 1,
    "currentPage": 1,
    "sort": "NEW",
    "moreParam.direction": "next" # 다음 페이지를 향해 간다는 방향 설정
}

print("커서 기반 댓글 수집을 시작합니다...")

while True:
    try:
        response = requests.get(url, headers=headers, params=params)
        text_data = response.text
        
        # JSONP 처리 (_callback 껍데기 벗기기)
        if text_data.startswith('_callback('):
            json_str = text_data.replace('_callback(', '')[:-2]
            data = json.loads(json_str)
        else:
            data = response.json()

        result = data.get('result', {})
        comments = result.get('commentList', [])
        page_model = result.get('pageModel', {}) # ⭐️ 페이지 이동 정보가 담긴 핵심 데이터
        
        # 종료 조건
        if not comments or comments == previous_comments:
            print(f"수집 완료! 더 이상 새로운 댓글이 없습니다. (총 {page-1}페이지 탐색)")
            break
            
        all_comments.extend(comments)
        previous_comments = comments
        print(f"{page}페이지 수집 완료... (누적: {len(all_comments)}개)")
        
        # ⭐️ 핵심 로직: 다음 요청을 위한 파라미터(커서) 업데이트
        # 서버가 알려준 다음 페이지 번호와 복잡한 고유 숫자(current, next 등)를 파라미터에 덮어씌웁니다.
        params['page'] = page + 1
        params['currentPage'] = page + 1
        
        if 'current' in page_model:
            params['current'] = page_model['current']
        if 'prev' in page_model:
            params['prev'] = page_model['prev']
        if 'next' in page_model:
            params['moreParam.next'] = page_model['next']
            
        page += 1
        time.sleep(0.5) 
        
    except Exception as e:
        print(f"오류 발생: {e}")
        break

df = pd.DataFrame(all_comments)
print(f"\n최종 수집된 댓글 수: {len(df)}개")

커서 기반 댓글 수집을 시작합니다...
1페이지 수집 완료... (누적: 20개)
수집 완료! 더 이상 새로운 댓글이 없습니다. (총 1페이지 탐색)

최종 수집된 댓글 수: 20개


In [ ]:
import requests
import time
import json
import pandas as pd


url = "https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://n.news.naver.com/article/001/0016305054" # 기사 원문 주소
}

all_comments = []
page = 1

print("댓글 수집을 시작합니다...")

while True:
    # ⭐️ 핵심: pageType을 more로 명시하고, page 숫자를 올립니다.
    params = {
        "ticket": "news",
        "templateId": "default_politics_m3",
        "pool": "cbox5",
        "lang": "ko",
        "country": "KR",
        "objectId": "news001,0016305054", 
        "pageSize": 20,
        "indexSize": 10,
        "listType": "OBJECT",
        "page": page, 
        "currentPage": page - 1 if page > 1 else 1, # 사용자가 직전에 보던 페이지 번호
        "pageType": "more", # ⭐️ 이것이 빠지면 네이버가 1페이지만 계속 줍니다!
        "sort": "NEW"
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        text_data = response.text.replace('_callback(', '')[:-2] # JSONP 껍데기 제거
        data = json.loads(text_data)

        # 댓글 리스트와 페이지 정보 추출
        result = data.get('result', {})
        comments = result.get('commentList', [])
        page_model = result.get('pageModel', {})
        total_pages = page_model.get('totalPages', 1) # 전체 페이지 수 파악 (예: 7)
            
        all_comments.extend(comments)
        print(f"{page}페이지 수집 완료... (누적: {len(all_comments)}개)")
        
        # ⭐️ 종료 조건: 현재 페이지가 서버가 알려준 전체 페이지(7)와 같거나 크면 종료
        if page >= total_pages:
            print(f"더 이상 페이지가 없습니다. (총 {total_pages}페이지 탐색 완료)")
            break
            
        page += 1
        time.sleep(0.5) 
        
    except Exception as e:
        print(f"오류 발생: {e}")
        break

# 데이터프레임 변환 및 정제
df = pd.DataFrame(all_comments)

if not df.empty:
    # 핵심 컬럼만 추출 (내용, 작성시간, 닉네임, 공감수, 비공감수)
    target_columns = ['contents', 'regTime', 'userName', 'sympathyCount', 'antipathyCount']
    available_columns = [col for col in target_columns if col in df.columns]
    df_clean = df[available_columns]
    
    print("\n[🎉 수집 완료!] -----------------")
    print(f"최종 수집된 댓글 수: {len(df_clean)}개")
    print(df_clean.head())
else:
    print("수집된 데이터가 없습니다.")

SyntaxError: invalid syntax (2634354728.py, line 65)